In [1]:
import Pkg
import Base.find_package

needed = ["Distributions", "Plots", "StatsBase", "QuantileRegressions"]

for pkg in needed
    if isnothing(find_package(pkg))
        println("Installing ", pkg, " …")
        Pkg.add(pkg)
    else
        println(pkg, " already installed.")
    end
end

Distributions already installed.
Plots already installed.
StatsBase already installed.
QuantileRegressions already installed.


In [ ]:
using Random, Distributions, Statistics, Printf, DelimitedFiles
using Plots

# ---------- helpers ----------
function clamp01(x)
    x < 0 ? 0.0 : (x > 1 ? 1.0 : x)
end

# a_t = 1 - (1 - ψ_t)^(1/α)
function baseline_alarm(psi, alpha)
    psi = clamp01(psi)
    alpha <= 0 && error("alpha must be > 0")
    1 - (1 - psi)^(1/alpha)
end

psi_memoryless(Istar_hist, N) = isempty(Istar_hist) ? 0.0 : Istar_hist[end] / N

function psi_sliding(Istar_hist, N, k_max)
    L = length(Istar_hist)
    L == 0 && return 0.0
    m = min(k_max, L)
    mean(@view(Istar_hist[(L - m + 1):L])) / N
end

function psi_powerlaw(Istar_hist, N, lambda_P)
    L = length(Istar_hist)
    L == 0 && return 0.0
    num  = 0.0
    @inbounds for j in 1:L
        w = j^(-lambda_P)
        num  += w * Istar_hist[end - j + 1]
    end
    num / N 
end

function psi_exponential(Istar_hist, N, lambda_E)
    L = length(Istar_hist)
    L == 0 && return 0.0
    num  = 0.0
    @inbounds for j in 0:(L - 1)
        w = exp(-lambda_E * j)
        num  += w * Istar_hist[end - j]
    end
    num / N   
end

function psi_reciprocal(Istar_hist, N, lambda_R)
    lambda_R <= 0 && error("lambda_R must be > 0")
    L = length(Istar_hist)
    L == 0 && return 0.0
    num = 0.0
    @inbounds for j in 0:(L - 1)
        w = 1.0 / (1.0 + lambda_R * j)   
        num += w * Istar_hist[end - j]
    end
    num / N
end

# ---------- simulate one epidemic ----------
function simulate_epidemic(N, I0, tau, beta, alpha, rateI;
                           mechanism::Symbol = :memoryless,
                           k_max::Union{Nothing,Int}=nothing,
                           lambda_P::Union{Nothing,Float64}=nothing,
                           lambda_E::Union{Nothing,Float64}=nothing,
                           lambda_R::Union{Nothing,Float64}=nothing)

    S      = zeros(Int, tau + 1)
    I      = zeros(Int, tau + 1)
    Istar  = zeros(Int, tau)
    Rstar  = zeros(Int, tau)
    psi    = zeros(Float64, tau)
    alarm  = zeros(Float64, tau)
    probSI = zeros(Float64, tau)
    probIR = 1 - exp(-rateI)

    S[1] = N - I0
    I[1] = I0

    # t = 1
    psi[1]   = 0.0
    alarm[1] = baseline_alarm(psi[1], alpha)
    pSI      = clamp01(1 - exp(-beta * (1 - alarm[1]) * (I[1] / N)))
    probSI[1] = pSI
    Istar[1] = rand(Binomial(S[1], pSI))
    Rstar[1] = rand(Binomial(I[1], probIR))
    S[2]     = S[1] - Istar[1]
    I[2]     = I[1] + Istar[1] - Rstar[1]

    # t = 2..tau
    for t in 2:tau
        hist = @view Istar[1:(t-1)]
        ψt = if mechanism === :memoryless
            psi_memoryless(hist, N)
        elseif mechanism === :sliding
            isnothing(k_max) && error("k_max must be provided for :sliding")
            psi_sliding(hist, N, k_max)
        elseif mechanism === :powerlaw
            isnothing(lambda_P) && error("lambda_P must be provided for :powerlaw")
            psi_powerlaw(hist, N, lambda_P)
        elseif mechanism === :exponential
            isnothing(lambda_E) && error("lambda_E must be provided for :exponential")
            psi_exponential(hist, N, lambda_E)
        elseif mechanism === :reciprocal
            isnothing(lambda_R) && error("lambda_R must be provided for :reciprocal")
            psi_reciprocal(hist, N, lambda_R)
        else
            error("Unknown mechanism: $mechanism")
        end
        psi[t]   = ψt
        alarm[t] = baseline_alarm(ψt, alpha)
        pSI      = clamp01(1 - exp(-beta * (1 - alarm[t]) * (I[t] / N)))
        probSI[t] = pSI

        Istar[t] = rand(Binomial(S[t], pSI))
        Rstar[t] = rand(Binomial(I[t], probIR))

        S[t+1] = S[t] - Istar[t]
        I[t+1] = I[t] + Istar[t] - Rstar[t]
    end

    return Dict(
        :Istar => Istar,
        :Rstar => Rstar,
        :S     => S,
        :I     => I,
        :psi   => psi,
        :alarm => alarm,
        :probSI => probSI,
        :probIR => probIR
    )
end

# ---------- simulate many ----------
function simulate_many(n_sims, N, I0, tau, beta, alpha, rateI;
                       mechanism::Symbol,
                       k_max::Union{Nothing,Int}=nothing,
                       lambda_P::Union{Nothing,Float64}=nothing,
                       lambda_E::Union{Nothing,Float64}=nothing,
                       lambda_R::Union{Nothing,Float64}=nothing,
                       min_total_incidence::Int = 100)

    mat = fill(NaN, n_sims, 3*tau)  # columns: Istar[1..tau], Rstar[1..tau], alarm[1..tau]
    keep = falses(n_sims)

    for s in 1:n_sims
        t0 = time()
        sim = simulate_epidemic(N, I0, tau, beta, alpha, rateI;
                                mechanism=mechanism,
                                k_max=k_max, lambda_P=lambda_P, lambda_E=lambda_E, lambda_R=lambda_R)
        t1 = time()
        @printf("Mechanism %s: finished simulation %d/%d in %.3f seconds.\n",
                String(mechanism), s, n_sims, t1 - t0)

        Istar = sim[:Istar]; Rstar = sim[:Rstar]; alarm = sim[:alarm]
        mat[s, 1:tau]                .= Istar
        mat[s, (tau+1):(2*tau)]      .= Rstar
        mat[s, (2*tau+1):(3*tau)]    .= alarm
        keep[s] = sum(Istar) > min_total_incidence
    end

    return Dict(:all => mat, :filtered => mat[keep, :])
end

# ---------- write CSV with header ----------
function write_matrix_csv(path::String, header::Vector{String}, M::AbstractMatrix)
    open(path, "w") do io
        println(io, join(header, ","))
        for i in 1:size(M,1)
            println(io, join(M[i, :], ","))
        end
    end
end

# ---------- plotting ----------
function plot_sim_set(toSave::AbstractMatrix, tau::Int, title_text::String, outfile::String)
    ns = size(toSave, 1)
    if ns == 0
        @info "No simulations to plot for $title_text"
        return
    end
    plt = plot(size=(1300,720), legend=false, title=title_text,
               xlabel="Epidemic Time", ylabel="Incidence")
    for s in 1:ns
        plot!(1:tau, toSave[s, 1:tau], alpha=0.25, lw=0.6)
    end
    savefig(plt, outfile)
end

function process_sim_results(toSave::AbstractMatrix, tau::Int, model_tag::String,
                             display_title::String; figs_dir="figs", data_dir="data")
    ns = size(toSave,1)
    ns == 0 && return

    Iblock   = toSave[:, 1:tau]
    alarmblk = toSave[:, (2*tau+1):(3*tau)]

    mean_Istar = vec(mean(Iblock, dims=1))
    mean_alarm = vec(mean(alarmblk, dims=1))

    # save CSVs
    isdir(data_dir) || mkpath(data_dir)
    inc_hdr = ["day","mean_Istar"]
    inc_mat = hcat(collect(1:tau), mean_Istar)
    write_matrix_csv(joinpath(data_dir, "$(model_tag)_mean_incidence.csv"), inc_hdr, inc_mat)

    alm_hdr = ["day","mean_alarm"]
    alm_mat = hcat(collect(1:tau), mean_alarm)
    write_matrix_csv(joinpath(data_dir, "$(model_tag)_mean_alarm.csv"), alm_hdr, alm_mat)

    # plots
    isdir(figs_dir) || mkpath(figs_dir)
    p1 = plot(1:tau, mean_Istar, lw=2, title=display_title,
              xlabel="Epidemic Time", ylabel="Mean incidence", size=(1300,720), legend=false)
    savefig(p1, joinpath(figs_dir, "$(model_tag)_mean.png"))

    p2 = plot(1:tau, mean_alarm, lw=2, title=display_title,
              xlabel="Epidemic Time", ylabel="Mean alarm", size=(1300,720), legend=false)
    savefig(p2, joinpath(figs_dir, "$(model_tag)_mean_alarm.png"))
end

# =============================
# Global settings (same as R)
# =============================
Random.seed!(1234)

nSim  = 20
N     = 1_000_000
I0    = 10
tau   = 50
rateI = 0.33
beta  = 0.8
alpha = 0.0044

k_max_true   = 14
lambda_P_tv  = 0.63
lambda_E_tv  = 0.35
lambda_R_tv  = 0.5  

isdir("data") || mkpath("data")
isdir("figs") || mkpath("figs")

# Column headers for saved simulation matrices
Ihdr   = ["Istar[" * string(i) * "]" for i in 1:tau]
Rhdr   = ["Rstar[" * string(i) * "]" for i in 1:tau]
Ahdr   = ["alarm[" * string(i) * "]" for i in 1:tau]
full_header = vcat(Ihdr, Rhdr, Ahdr)

# 1) Memoryless
res_mem = simulate_many(1000, N, I0, tau, beta, alpha, rateI; mechanism=:memoryless)
toSave = res_mem[:filtered]
if size(toSave,1) > nSim
    toSave = toSave[1:nSim, :]
end
write_matrix_csv("data/memoryless.csv", full_header, toSave)
plot_sim_set(toSave, tau, "Memoryless", "figs/memoryless.png")
process_sim_results(toSave, tau, "memoryless", "Memoryless")

# 2) Sliding window (k_max = 14)
res_win = simulate_many(1000, N, I0, tau, beta, alpha, rateI; mechanism=:sliding, k_max=k_max_true)
toSave = res_win[:filtered]
if size(toSave,1) > nSim
    toSave = toSave[1:nSim, :]
end
write_matrix_csv("data/sliding_kmax14.csv", full_header, toSave)
plot_sim_set(toSave, tau, "Sliding window", "figs/sliding_kmax14.png")
process_sim_results(toSave, tau, "sliding_kmax14", "Sliding window")

# 3) Power-law decay (lambda_P)
res_pow = simulate_many(1000, N, I0, tau, beta, alpha, rateI; mechanism=:powerlaw, lambda_P=lambda_P_tv)
toSave = res_pow[:filtered]
if size(toSave,1) > nSim
    toSave = toSave[1:nSim, :]
end
write_matrix_csv("data/powerlaw_lambdaP.csv", full_header, toSave)
plot_sim_set(toSave, tau, "Power-law", "figs/powerlaw_lambdaP.png")
process_sim_results(toSave, tau, "powerlaw_lambdaP", "Power-law")

# 4) Exponential decay (lambda_E)
res_exp = simulate_many(1000, N, I0, tau, beta, alpha, rateI; mechanism=:exponential, lambda_E=lambda_E_tv)
toSave = res_exp[:filtered]
if size(toSave,1) > nSim
    toSave = toSave[1:nSim, :]
end
write_matrix_csv("data/exponential_lambdaE.csv", full_header, toSave)
plot_sim_set(toSave, tau, "Exponential", "figs/exponential_lambdaE.png")
process_sim_results(toSave, tau, "exponential_lambdaE", "Exponential")

# 5) Reciprocal decay (lambda_R)
res_rec = simulate_many(1000, N, I0, tau, beta, alpha, rateI; mechanism=:reciprocal, lambda_R=lambda_R_tv)
toSave = res_rec[:filtered]
if size(toSave,1) > nSim
    toSave = toSave[1:nSim, :]
end
write_matrix_csv("data/reciprocal_lambdaR.csv", full_header, toSave)
plot_sim_set(toSave, tau, "Reciprocal", "figs/reciprocal_lambdaR.png")
process_sim_results(toSave, tau, "reciprocal_lambdaR", "Reciprocal")

println("Done.")


Mechanism memoryless: finished simulation 1/1000 in 0.275 seconds.
Mechanism memoryless: finished simulation 2/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 3/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 4/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 5/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 6/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 7/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 8/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 9/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 10/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 11/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 12/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 13/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 14/1000 in 0.000 seconds.
Mechanism memoryless: finished simulation 15/1000 in 0.00